# Imports and reads

# Imports and reads

In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt
from utils.data_loader import load_parquet, find_repo_root
from pathlib import Path

# Load only specific columns that you need for analysis
# This will significantly reduce memory usage
columns_needed = ['time', 'orbital_decay', 'F10.7 (LASP)', 'Kp (LASP)']

GFOC_data = load_parquet(columns=columns_needed)
print(GFOC_data.head())

✅ Successfully loaded data with shape: (3157920, 4)
                 time  orbital_decay  F10.7 (LASP)  Kp (LASP)
0 2023-01-01 00:00:00        16.6730         153.0      2.333
1 2023-01-01 00:00:20        16.6730         153.0      2.333
2 2023-01-01 00:00:40        16.6735         153.0      2.333
3 2023-01-01 00:01:00        16.6740         153.0      2.333
4 2023-01-01 00:01:20        16.6740         153.0      2.333


In [2]:
df = GFOC_data.copy()
time = df['time'].values

# make 'time' the index
df.set_index('time', inplace=True)
df.index = pd.to_datetime(df.index)
dt_seconds = (df.index[1] - df.index[0]).total_seconds()

# Import Subsets

In [3]:
def load_intervals(csv_paths):
    """Load start/end intervals from one or more CSV files."""
    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path, parse_dates=["start", "end"])
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

def merge_intervals(df):
    """Merge overlapping time intervals."""
    # Sort intervals by start time
    df = df.sort_values("start").reset_index(drop=True)

    merged = []
    current_start = df.loc[0, "start"]
    current_end = df.loc[0, "end"]

    for i in range(1, len(df)):
        row_start = df.loc[i, "start"]
        row_end = df.loc[i, "end"]

        if row_start <= current_end:  
            # Overlapping or touching intervals → extend the end if needed
            current_end = max(current_end, row_end)
        else:  
            # No overlap → push current and reset
            merged.append((current_start, current_end))
            current_start = row_start
            current_end = row_end

    # Append final interval
    merged.append((current_start, current_end))

    return pd.DataFrame(merged, columns=["start", "end"])


In [6]:
interval_eflag = pd.read_csv(find_repo_root() / Path("Analysis/Subsets/subsets_eflag.csv"), parse_dates=["start", "end"])
interval_Kp = pd.read_csv(find_repo_root() / Path("Analysis/Subsets/subsets_Kp.csv"), parse_dates=["start", "end"])
interval_meanstd = pd.read_csv(find_repo_root() / Path("Analysis/Subsets/subsets_meanstd.csv"), parse_dates=["start", "end"])


# add timedelta to meanstd
t01 = pd.Timedelta(hours=72)  # extension before 0->1
t10 = pd.Timedelta(hours=12)  # extension after 1->0
interval_meanstd['start'] = interval_meanstd['start'] - t01
interval_meanstd['end'] = interval_meanstd['end'] + t10

# concatenate all intervals
intervals_df = pd.concat([interval_eflag, interval_meanstd], ignore_index=True) #, interval_Kp
# merge
merged_df = merge_intervals(intervals_df)
# save merged intervals
merged_df.to_csv(find_repo_root() / Path("Analysis/Subsets/subsets_eflag_meanstd.csv"), index=False)
print(f"# subsets: {len(merged_df)}")

# subsets: 58
